# Week 4: Baseline Model

============================================================

IDX Exchange — Data Science Internship

Week 4: Baseline Model

Author: Zachary Blehm

============================================================

Summary:

In [ ]:
# Imports and File Loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
# Function to split the most recent n_months from the training set
# This function allows for using n_months a parameter
def keep_recent_months(df, n_months):
    df = df.copy()

    # Create a year-month period
    year_month = pd.PeriodIndex.from_fields(
        year=df["CloseYear"].astype(int),
        month=df["CloseMonth"].astype(int),
        freq="M"
    )

    df["_YearMonth"] = year_month

    # Identify the most recent month in the dataset
    latest_month = df["_YearMonth"].max()

    # Calculate the earliest month to retain (inclusive)
    earliest_month = latest_month - (n_months - 1)

    # Keep the most recent N calendar months
    df = df[
        (df["_YearMonth"] >= earliest_month) &
        (df["_YearMonth"] <= latest_month)
    ].copy()

    # Remove the temporary column
    df.drop(columns="_YearMonth", inplace=True)

    return df

In [ ]:
lr_evaluation_log = []

def train_and_evaluate_LR(X, y, X_eval, y_eval, description, num_months = None):
    if num_months == None:
        # Create a year-month period
        year_month = pd.PeriodIndex.from_fields(
            year=df["CloseYear"].astype(int),
            month=df["CloseMonth"].astype(int),
            freq="M"
        )

        X["_YearMonth"] = year_month

        # Identify the most recent month in the dataset
        latest_month = X["_YearMonth"].max()
        earliest_month = X["_YearMonth"].min()

        # Calculate the earliest month to retain (inclusive)
        num_months = latest_month - earliest_month

        # Remove the temporary column
        X.drop(columns="_YearMonth", inplace=True)

    # Train model
    model = LinearRegression()
    model.fit(X, y)

    # Predict
    y_eval_pred = model.predict(X_eval)

    # Evaluate
    r2 = r2_score(y_eval, y_eval_pred)

    lr_evaluation_log.append({
          "Description of Data": description,
          "Training Period": num_months,
          "R-squared": r2
    })

## Create Train Splits - Full Feature Set

We treat the number of months of training data used as a tunable parameter and use the validation set for evaulation. But we note that the preprocessing was done using the full 26 month training set. In other words imputation via aggregates is based on the aggregate of 26 months (not the different windows). Similarly, outlier removal based on percentile was calculated from the full 26 months of data (again, not the different windows).

Additionally we consider four different feature sets. First the full feature set as exported from 02_preprocessing.ipynb, a subset of medium to high correlation features, a subset of high correlation features and finally a minimum feature set of living area, lot size and number of bedrooms and bathrooms.

In [ ]:
# Training data is 26 months total
# Split training at 6, 12, 18, 24
# Validation is 202603, Test is 202604

df_train_full = pd.read_csv("train_26mo.csv")
df_val = pd.read_csv("val_26mo.csv")
df_test = pd.read_csv("test_26mo.csv")

In [ ]:
# validation set
X_val = df_val.drop(columns=["ClosePrice"])
y_val = df_val["ClosePrice"]

In [ ]:
months = [6, 9, 12, 15, 18, 21, 24]

def train_by_num_months(df, df_eval, features,  months, description):
    for num_months in months:
        df_train = keep_recent_months(df, num_months)

        X = df_train[features]
        y = df_train["ClosePrice"]

        X_eval = df_val[features]
        y_eval = df_val["ClosePrice"]

        train_and_evaluate_LR(X, y, X_eval, y_eval, description, num_months)

        print(lr_evaluation_log[len(lr_evaluation_log) - 1])

In [ ]:
full_feature_set = df_train_full.drop(columns=["ClosePrice"]).columns.tolist()

train_by_num_months(df_train_full,
                    df_val,
                    full_feature_set,
                    months,
                    "Full Feature Set")

{'Description of Data': 'Full Feature Set', 'Training Period': 6, 'R-squared': 0.8620160979266591}
{'Description of Data': 'Full Feature Set', 'Training Period': 9, 'R-squared': 0.8635591375721016}
{'Description of Data': 'Full Feature Set', 'Training Period': 12, 'R-squared': 0.8636391705863731}
{'Description of Data': 'Full Feature Set', 'Training Period': 15, 'R-squared': 0.8639259244375747}
{'Description of Data': 'Full Feature Set', 'Training Period': 18, 'R-squared': 0.863856328067243}
{'Description of Data': 'Full Feature Set', 'Training Period': 21, 'R-squared': 0.8640753264102983}
{'Description of Data': 'Full Feature Set', 'Training Period': 24, 'R-squared': 0.8643037566763624}


In [ ]:
high_to_medium_corr_features = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "knn5_median_price",
    "zip_price_trend",
    "Levels_missing",
    "MainLevelBedrooms_missing",
    "FireplaceYN"
    ""
]

train_by_num_months(df_train_full,
                    df_val,
                    high_to_medium_corr_features,
                    months,
                    "High to Medium Correlation Feature Set")

{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 6, 'R-squared': 0.8177489749261011}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 9, 'R-squared': 0.8182813590304453}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 12, 'R-squared': 0.8190144779216886}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 15, 'R-squared': 0.8190989127036428}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 18, 'R-squared': 0.8191737103021985}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 21, 'R-squared': 0.8193017431293856}
{'Description of Data': 'High to Medium Correlation Feature Set', 'Training Period': 24, 'R-squared': 0.8194163869220357}


In [ ]:
high_corr_features = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "knn5_median_price",
    "zip_price_trend"
]

train_by_num_months(df_train_full,
                    df_val,
                    high_corr_features,
                    months,
                    "High Correlation Feature Set")

{'Description of Data': 'High Correlation Feature Set', 'Training Period': 6, 'R-squared': 0.8163076287060533}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 9, 'R-squared': 0.8167803635566847}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 12, 'R-squared': 0.8175252488855824}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 15, 'R-squared': 0.8175508286438697}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 18, 'R-squared': 0.8175503474099362}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 21, 'R-squared': 0.8176445544496311}
{'Description of Data': 'High Correlation Feature Set', 'Training Period': 24, 'R-squared': 0.8177112602882018}


In [ ]:
primary_features = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "LotSizeSquareFeet"
]

train_by_num_months(df_train_full,
                    df_val,
                    primary_features,
                    months,
                    "4 Primary Features (living/lot are & num bed/bath)")

{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 6, 'R-squared': 0.3232121501524716}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 9, 'R-squared': 0.3242999802860551}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 12, 'R-squared': 0.3245253611674105}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 15, 'R-squared': 0.3244355218554805}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 18, 'R-squared': 0.32452723526464755}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 21, 'R-squared': 0.3244337243579397}
{'Description of Data': '4 Primary Features (living/lot are & num bed/bath)', 'Training Period': 24, 'R-squared': 0.3241665845199905}


#TODO

Consider shorter timeframes in the preprocessing step. This will affect some features, for example the KNN aggregates.